# Notebook 02 — Drug Comparison

Compares the four compounds in the drug library across:
- Brain concentration at equivalent oral dose
- D1 / D2 receptor occupancy
- Circuit steady-state changes
- AND-gate consolidation probability

**Drugs**: Haloperidol · Cocaine · L-DOPA · Amphetamine


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from neuropharm_sim.simulation import SimulationRunner
from neuropharm_sim.drug_library import list_drugs, get_drug
from neuropharm_sim.visualization import plot_dose_response_comparison

## 1. Drug library overview

In [ ]:
drugs = ["haloperidol", "cocaine", "ldopa", "amphetamine"]
rows = []
for name in drugs:
    d = get_drug(name)
    rows.append({
        "Drug": d.name,
        "Ki D1 (nM)": d.ki_d1_nm,
        "Ki D2 (nM)": d.ki_d2_nm,
        "DAT block": f"{d.dat_block_fraction*100:.0f}%",
        "DA release fold": d.dat_release_fold,
        "t½ (h)": d.half_life_h,
        "F (%)": f"{d.bioavailability*100:.0f}%",
    })
pd.DataFrame(rows).set_index("Drug")

## 2. Simulate each drug at 5 mg

In [ ]:
runner = SimulationRunner(t_simulate_ms=1000)
results = runner.compare_drugs(drugs, dose_mg=5.0)

for name, res in results.items():
    print("─" * 50)
    print(res.summary())
print("─" * 50)

## 3. Summary table

In [ ]:
summary_rows = []
for name, res in results.items():
    gate_p = res.consolidation.mean_gate_probability if res.consolidation else float("nan")
    summary_rows.append({
        "Drug": res.drug_name,
        "Brain conc (nM)": f"{res.brain_concentration_nm:.3f}",
        "D2 occ (%)": f"{res.d2_occupancy*100:.1f}",
        "D1 occ (%)": f"{res.d1_occupancy*100:.1f}",
        "VTA rate": f"{res.steady_state['r_vta']:.3f}",
        "NAcD1 rate": f"{res.steady_state['r_nacd1']:.3f}",
        "AND-gate P": f"{gate_p:.4f}",
    })

pd.DataFrame(summary_rows).set_index("Drug")

## 4. D2 occupancy dose-response

In [ ]:
doses = np.linspace(0.5, 15, 25)

results_by_drug = {}
for name in ["haloperidol", "cocaine", "amphetamine"]:
    results_by_drug[name] = runner.dose_response(name, doses, evaluate_and_gate=False)

fig, ax = plot_dose_response_comparison(
    results_by_drug,
    metric="d2_occupancy",
    ylabel="D2 receptor occupancy (%)",
)
plt.show()

## 5. Circuit trajectories side by side

In [ ]:
from neuropharm_sim.visualization import plot_circuit_dynamics

drug_colors = {
    "haloperidol": "#2166ac",
    "cocaine":     "#d6604d",
    "ldopa":       "#4dac26",
    "amphetamine": "#f4a582",
}

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
node_keys = ["r_vta", "r_nacd1", "r_nacd2", "r_pfc"]
node_labels = ["VTA", "NAcD1", "NAcD2", "PFC"]

for name, res in results.items():
    t = res.circuit_trajectory["t_ms"]
    for ax, key in zip(axes, node_keys):
        ax.plot(t, res.circuit_trajectory[key], lw=1.8,
                label=res.drug_name, color=drug_colors.get(name, None), alpha=0.85)

for ax, label in zip(axes, node_labels):
    ax.set_ylabel(label)
    ax.set_ylim(0, 1)
    ax.set_ylabel(f"{label}\nfiring rate")

axes[-1].set_xlabel("Time (ms)")
axes[0].legend(ncol=4, fontsize=9, loc="upper right")
fig.suptitle("Circuit Trajectories: Four Drugs at 5 mg", fontweight="bold")
fig.tight_layout()
plt.show()